# 01 - Susceptibility model comparison

**SCAFFOLD - NOT RUN.** Requires an approved real event inventory and covariate dataset. Synthetic labels are for code tests only.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%pip install -q pandas numpy scikit-learn xgboost lightgbm mlflow

import random
import numpy as np
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

In [ ]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path('/content/drive/MyDrive/terrasafe/labelled_inventory.csv')  # Set to reviewed data
FEATURES = ['elevation_m', 'slope_deg', 'aspect_sin', 'aspect_cos', 'curvature', 'twi', 'ndvi', 'soil_clay_fraction', 'land_cover_code', 'road_distance_km', 'stream_distance_km']
LABEL = 'landslide_label'
BLOCK = 'spatial_block'
cells = pd.read_csv(DATA_PATH)
required = set(FEATURES + [LABEL, BLOCK, 'label_source'])
missing = required - set(cells.columns)
if missing:
    raise ValueError(f'Missing reviewed data columns: {sorted(missing)}')
if cells.get('synthetic', pd.Series(False, index=cells.index)).astype(bool).all():
    raise ValueError('Synthetic-only labels cannot be used for real model evaluation')
assert cells[LABEL].nunique() == 2
X, y, groups = cells[FEATURES], cells[LABEL], cells[BLOCK]

In [ ]:
from sklearn.model_selection import GroupKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

cv = GroupKFold(n_splits=5)
models = {
    'LogisticRegression': make_pipeline(StandardScaler(), LogisticRegression(class_weight='balanced', random_state=SEED, max_iter=1000)),
    'RandomForest': RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=SEED, n_jobs=1),
    'XGBoost': XGBClassifier(n_estimators=250, max_depth=4, eval_metric='logloss', random_state=SEED, n_jobs=1),
    'LightGBM': LGBMClassifier(n_estimators=250, class_weight='balanced', random_state=SEED, verbosity=-1, n_jobs=1),
}
scoring = {'precision': 'precision', 'recall': 'recall', 'f1': 'f1', 'roc_auc': 'roc_auc', 'average_precision': 'average_precision'}
results = {name: cross_validate(model, X, y, groups=groups, cv=cv, scoring=scoring, return_estimator=False) for name, model in models.items()}
# Build a stacked candidate only after the same spatial folds; keep it only if held-out metrics improve and miss rate remains acceptable.

In [ ]:
# Spatially nested stacking scaffold: generate base-model out-of-fold predictions
# within each outer training fold; never use outer test blocks for meta-model fitting.
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

outer = GroupKFold(n_splits=5)
stacked_fold_auc = []
for outer_train, outer_test in outer.split(X, y, groups):
    X_train, X_test = X.iloc[outer_train], X.iloc[outer_test]
    y_train, y_test = y.iloc[outer_train], y.iloc[outer_test]
    g_train = groups.iloc[outer_train]
    inner = GroupKFold(n_splits=3)
    oof = np.zeros((len(outer_train), len(models)))
    for inner_train, inner_valid in inner.split(X_train, y_train, g_train):
        for model_index, prototype in enumerate(models.values()):
            fitted = clone(prototype).fit(X_train.iloc[inner_train], y_train.iloc[inner_train])
            oof[inner_valid, model_index] = fitted.predict_proba(X_train.iloc[inner_valid])[:, 1]
    meta = LogisticRegression(class_weight='balanced', random_state=SEED, max_iter=1000).fit(oof, y_train)
    outer_probabilities = []
    for prototype in models.values():
        fitted = clone(prototype).fit(X_train, y_train)
        outer_probabilities.append(fitted.predict_proba(X_test)[:, 1])
    stacked_probability = meta.predict_proba(np.column_stack(outer_probabilities))[:, 1]
    stacked_fold_auc.append(roc_auc_score(y_test, stacked_probability))
# Compare these outer-fold results with the best single model using the same folds.
# Keep the ensemble only if its held-out improvement is repeatable and miss-rate tradeoffs are accepted.

No estimator comparison or ensemble selection has been run. Record data provenance, spatial-block design, false-negative rate, and independent temporal test results before discussing deployment.